## Imports

In [0]:
from pyspark.sql import SparkSession
import pandas as pd

## Get the needed Parameters

In [0]:
run_date = dbutils.widgets.get("run_date")

Businessdate = spark.sql(f"""                    
                        select 
                            case 
                                when try_cast({run_date!r} as date) is not null then try_cast({run_date!r} as date)
                                else PreviousBusinessDay
                            end as BusinessDate
                        from `p-cdh`.reference.ref_calendar
                        where CalendarDate = coalesce(try_cast({run_date!r} as date),cast(getdate() as date))
                    """).first()[0]

print(Businessdate)

## Get the needed Source Tables

In [0]:
Portfolio = spark.sql(f"""                   
                            SELECT '31D'    AS PortfolioKey
                                 , 'DMW'     AS Prefix
                            UNION          
                            SELECT '30D'    AS PortfolioKey
                                 , 'DMR'     AS Prefix
                            UNION          
                            SELECT '3031'      AS PortfolioKey
                                 , 'DMD'     AS Prefix
                        """)

Portfolio.createOrReplaceTempView("Portfolio")
display(Portfolio)

### Get Profits Source Tables

In [0]:
loan_details = spark.sql(f"""                    
                            select distinct Account_Number
                            from `p-cdh`.landed.profits_loan_details
                            where BusinessDate = '{Businessdate}'
                        """)

loan_details.createOrReplaceTempView("loan_details")
# display(loan_details)

In [0]:
customer = spark.sql(f"""                    
                        select distinct cdi, cust_id
                        from `p-cdh`.landed.profits_customer
                        where BusinessDate = '{Businessdate}'
                    """)

customer.createOrReplaceTempView("customer")
# display(customer)

### Get Arotron Source Tables

In [0]:
ArxAgent = spark.sql(f"""                    
                        select distinct UserId, UserName
                        from `p-cdh`.landed.arotron_ArxAgent
                        where BusinessDate = '{Businessdate}'
                    """)

ArxAgent.createOrReplaceTempView("ArxAgent")
# display(ArxAgent)

In [0]:
ArxAssign = spark.sql(f"""                    
                        select distinct CustCDI, UserId, AsnType, AsnEndDT
                        from `p-cdh`.landed.arotron_ArxAssign
                        where BusinessDate = '{Businessdate}'
                    """)

ArxAssign.createOrReplaceTempView("ArxAssign")
# display(ArxAssign)

In [0]:
eaiSIG_BP = spark.sql(f"""                    
                        select distinct eaisig_bp_cust_s_unid
                        from `p-cdh`.landed.arotron_eaiSIG_BP
                        where BusinessDate = '{Businessdate}'
                    """)

eaiSIG_BP.createOrReplaceTempView("eaiSIG_BP")
# display(eaiSIG_BP)

In [0]:
eaiSIG_BE = spark.sql(f"""                    
                        select distinct eaiSIG_BE_CASE_S_ID
                        from `p-cdh`.landed.arotron_eaiSIG_BE
                        where BusinessDate = '{Businessdate}'
                    """)

eaiSIG_BE.createOrReplaceTempView("eaiSIG_BE")
# display(eaiSIG_BE)

In [0]:
eaiSIG_MOD = spark.sql(f"""                    
                        select distinct eaiSIG_MOD_CASE_S_ID, eaiSIG_MOD_OLD_ACCOUNT_ID
                        from `p-cdh`.landed.arotron_eaiSIG_MOD
                        where BusinessDate = '{Businessdate}'
                    """)

eaiSIG_MOD.createOrReplaceTempView("eaiSIG_MOD")
# display(eaiSIG_MOD)

In [0]:
eaiSIG_BPR = spark.sql(f"""                    
                        select distinct eaiSIG_BPR_CUST_S_UNID, eaiSIG_Bpr_CASE_S_ID
                        from `p-cdh`.landed.arotron_eaiSIG_BPR
                        where BusinessDate = '{Businessdate}'
                    """)

eaiSIG_BPR.createOrReplaceTempView("eaiSIG_BPR")
# display(eaiSIG_BPR)

In [0]:
arxLegAct = spark.sql(f"""                    
                        select distinct custcdi, contNum
                        from `p-cdh`.landed.arotron_arxLegAct
                        where BusinessDate = '{Businessdate}'
                    """)

arxLegAct.createOrReplaceTempView("arxLegAct")
# display(arxLegAct)

In [0]:
ArxLegExp = spark.sql(f"""                    
                        select distinct custcdi
                        from `p-cdh`.landed.arotron_ArxLegExp
                        where BusinessDate = '{Businessdate}'
                    """)

ArxLegExp.createOrReplaceTempView("ArxLegExp")
# display(ArxLegExp)

## Check Tables

### Customers

#### Check in prperson

In [0]:
NumberOfPersons = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct crs)
                                from prperson prp
                                inner join Portfolio port
                                on port.Prefix = left(prp.crs,3)
                                group by all
                            """)

NumberOfPersons.createOrReplaceTempView("NumberOfPersons")
display(NumberOfPersons)

In [0]:
ComparisonOfPersons = spark.sql(f"""                    
                                    select port.Prefix, crs
                                    from prperson prp
                                    inner join Portfolio port
                                    on port.Prefix = left(prp.crs,3)
                                    except
                                    select port.Prefix, cdi
                                    from customer cust
                                    inner join Portfolio port
                                    on port.Prefix = left(cust.cdi,3)
                                """)

ComparisonOfPersons.createOrReplaceTempView("ComparisonOfPersons")
display(ComparisonOfPersons)